<a href="https://colab.research.google.com/github/jpegiel/TPX3/blob/main/BB84_Cirq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install cirq --quiet

import cirq
import cirq_web
from random import choices
import numpy as np
from typing import List, Tuple, Dict

Helper functions for message encryption

In [ ]:
def text_to_bits(message: str) -> List[int]:
    """Convert text message to list of bits (ASCII encoding)"""
    bits = []
    for char in message:
        ascii_val = ord(char)
        binary_str = format(ascii_val, '08b')
        bits.extend([int(b) for b in binary_str])
    return bits

def bits_to_text(bits: List[int]) -> str:
    """Convert list of bits back to text message"""
    message = ""
    for i in range(0, len(bits), 8):
        byte_bits = bits[i:i+8]
        if len(byte_bits) == 8:
            ascii_val = int(''.join(map(str, byte_bits)), 2)
            message += chr(ascii_val)
    return message

def xor_encrypt_decrypt(message_bits: List[int], key_bits: List[int]) -> List[int]:
    """XOR encryption/decryption (one-time pad)"""
    return [message_bits[i] ^ key_bits[i] for i in range(len(message_bits))]

def compare_keys(key1: List[int], key2: List[int]) -> float:
    """Calculate percentage agreement between two keys"""
    if len(key1) != len(key2):
        return 0.0
    matches = sum(1 for a, b in zip(key1, key2) if a == b)
    return (matches / len(key1)) * 100

# BB84 Protocol Core Implementation

In [ ]:

# Define encoding and basis gates
encode_gates = {0: cirq.I, 1: cirq.X}
basis_gates = {'Z': cirq.I, 'X': cirq.H}

def run_bb84(num_bits: int, with_eve: bool = False, eve_attack_type: str = 'measure') -> Tuple:
    """
    Run the BB84 protocol simulation.

    Parameters:
    - num_bits: Number of qubits to send
    - with_eve: Whether to include Eve's eavesdropping
    - eve_attack_type: 'measure' for measurement attack, 'intercept' for intercept-resend

    Returns:
    Dictionary containing all keys and results
    """

    # Create qubits
    qubits = cirq.NamedQubit.range(num_bits, prefix='q')

    # Alice chooses random bits and bases
    alice_key = choices([0, 1], k=num_bits)
    alice_bases = choices(['Z', 'X'], k=num_bits)

    print(f"Alice's key (first 10 bits): {alice_key[:10]}...")
    print(f"Alice's bases (first 10): {alice_bases[:10]}...")

    # Alice creates and sends qubits
    alice_circuit = cirq.Circuit()

    for bit in range(num_bits):
        encode_gate = encode_gates[alice_key[bit]]
        basis_gate = basis_gates[alice_bases[bit]]
        qubit = qubits[bit]

        alice_circuit.append(encode_gate(qubit))
        alice_circuit.append(basis_gate(qubit))

    #EVE'S ATTACK (if present)
    eve_key = None
    eve_bases = None

    if with_eve:
        print(f"\n EVE IS EAVESDROPPING! Attack type: {eve_attack_type}")

        if eve_attack_type == 'measure':
            # Simple measurement attack - Eve measures immediately
            eve_circuit = cirq.Circuit()
            eve_circuit.append(cirq.measure(qubits, key='eve_key'))

            sim = cirq.Simulator()
            results = sim.run(alice_circuit + eve_circuit)
            eve_key = results.measurements['eve_key'][0]

            # Eve resends the qubits in their measured state (no basis info)
            # This corrupts any qubits where she measured in wrong basis
            alice_circuit = cirq.Circuit()
            for bit in range(num_bits):
                encode_gate = encode_gates[eve_key[bit]]
                alice_circuit.append(encode_gate(qubits[bit]))

        elif eve_attack_type == 'intercept':
            # Intercept-resend attack - Eve guesses bases
            eve_bases = choices(['Z', 'X'], k=num_bits)
            print(f"Eve's guessed bases (first 10): {eve_bases[:10]}...")

            # Eve applies her basis gates and measures
            eve_circuit = cirq.Circuit()
            for bit in range(num_bits):
                basis_gate = basis_gates[eve_bases[bit]]
                eve_circuit.append(basis_gate(qubits[bit]))
            eve_circuit.append(cirq.measure(qubits, key='eve_key'))

            sim = cirq.Simulator()
            results = sim.run(alice_circuit + eve_circuit)
            eve_key = results.measurements['eve_key'][0]

            # Eve resends new qubits based on her measurement results
            alice_circuit = cirq.Circuit()
            for bit in range(num_bits):
                encode_gate = encode_gates[eve_key[bit]]
                basis_gate = basis_gates[eve_bases[bit]]
                alice_circuit.append(encode_gate(qubits[bit]))
                alice_circuit.append(basis_gate(qubits[bit]))

    # Bob chooses random bases
    bob_bases = choices(['Z', 'X'], k=num_bits)
    print(f"Bob's bases (first 10): {bob_bases[:10]}...")

    # Bob applies his bases and measures
    bob_circuit = cirq.Circuit()

    for bit in range(num_bits):
        basis_gate = basis_gates[bob_bases[bit]]
        bob_circuit.append(basis_gate(qubits[bit]))

    bob_circuit.append(cirq.measure(qubits, key='bob_key'))

    # Run the full circuit
    full_circuit = alice_circuit + bob_circuit
    sim = cirq.Simulator()
    results = sim.run(full_circuit)
    bob_key = results.measurements['bob_key'][0]

    #Compare bases (keep only matching bases)
    final_alice_key = []
    final_bob_key = []
    final_eve_key = [] if with_eve else None

    for bit in range(num_bits):
        if alice_bases[bit] == bob_bases[bit]:
            final_alice_key.append(alice_key[bit])
            final_bob_key.append(bob_key[bit])
            if with_eve and eve_key is not None:
                final_eve_key.append(eve_key[bit])

    return {
        'alice_original': alice_key,
        'bob_original': bob_key,
        'eve_original': eve_key,
        'alice_final': final_alice_key,
        'bob_final': final_bob_key,
        'eve_final': final_eve_key,
        'alice_bases': alice_bases,
        'bob_bases': bob_bases,
        'eve_bases': eve_bases
    }

## Experiment 1 - Alice sends encrypted message to Bob (NO EVE)

In [ ]:
# Original message to send
original_message = "FOREST"
print(f"\n Original message: '{original_message}'")

# Convert message to bits
message_bits = text_to_bits(original_message)
print(f"Message as bits: {message_bits}")

# Run BB84 to generate key (need enough bits for the message)
num_qubits = len(message_bits) * 3  # Extra bits for basis comparison
results = run_bb84(num_qubits, with_eve=False)

# Use the final key (after basis comparison)
shared_key = results['alice_final'][:len(message_bits)]
print(f"\n Shared secret key (first {len(message_bits)} bits): {shared_key}")

# Encrypt the message
encrypted_bits = xor_encrypt_decrypt(message_bits, shared_key)
print(f" Encrypted bits: {encrypted_bits}")

# Decrypt the message
decrypted_bits = xor_encrypt_decrypt(encrypted_bits, shared_key)
decrypted_message = bits_to_text(decrypted_bits)
print(f" Decrypted message: '{decrypted_message}'")

# Verify success
print("\n" + "-"*50)
if original_message == decrypted_message:
    print(" SUCCESS! Message transmitted securely!")
    print(f"   Key agreement: {compare_keys(results['alice_final'], results['bob_final']):.1f}%")
else:
    print(" FAILURE! Message corrupted during transmission")
print("-"*50)

##  Experiment 3 - With Eve performing Intercept-Resend Attack

In [ ]:
original_message = "FOREST"
print(f"\n Original message: '{original_message}'")

message_bits = text_to_bits(original_message)
print(f"Message length: {len(message_bits)} bits")

# Run BB84 with Eve doing intercept-resend attack
num_qubits = len(message_bits) * 3  # More qubits needed due to higher error rate
results = run_bb84(num_qubits, with_eve=True, eve_attack_type='intercept')

# Use the final keys
shared_key_alice = results['alice_final'][:len(message_bits)]
shared_key_bob = results['bob_final'][:len(message_bits)]

key_match_rate = compare_keys(shared_key_alice, shared_key_bob)
print(f"\n Key agreement between Alice and Bob: {key_match_rate:.1f}%")

# Encrypt and decrypt
encrypted_bits = xor_encrypt_decrypt(message_bits, shared_key_alice)
decrypted_bits = xor_encrypt_decrypt(encrypted_bits, shared_key_bob)
decrypted_message = bits_to_text(decrypted_bits)

print(f"\n Bob's decrypted message: '{decrypted_message}'")

print("\n" + "-"*50)
if original_message == decrypted_message:
    print(" Message recovered despite Eve's attack!")
else:
    print(" Message corrupted! Eve's eavesdropping detected!")
    print(f"   Error rate introduced by Eve: {100 - key_match_rate:.1f}%")

    # Theoretical prediction: Eve gets caught ~25% of the time when bases don't match
    theoretical_caught_rate = 0.25 * 100
    print(f"   Theoretical detection probability: {theoretical_caught_rate:.0f}%")
print("-"*50)

# Show Eve's success rate
if results['eve_final']:
    eve_key = results['eve_final'][:len(message_bits)]
    eve_match_alice = compare_keys(eve_key, shared_key_alice)
    print(f"\n Eve successfully learned {eve_match_alice:.1f}% of the key bits")

## Experiment 4 - Statistical Analysis of Eve's Detection Probability

In [ ]:
import cirq
from random import choices
import numpy as np
from typing import List, Tuple, Dict

# Define encoding and basis gates
encode_gates = {0: cirq.I, 1: cirq.X}
basis_gates = {'Z': cirq.I, 'X': cirq.H}

def run_bb84(num_bits: int, with_eve: bool = False) -> Tuple:
    """
    Run the BB84 protocol simulation with the Intercept-Resend attack if Eve is present.

    Parameters:
    - num_bits: Number of qubits to send
    - with_eve: Whether to include Eve's eavesdropping

    Returns:
    Dictionary containing all keys and results
    """

    # Create qubits
    qubits = cirq.NamedQubit.range(num_bits, prefix='q')

    # Alice chooses random bits and bases
    alice_key = choices([0, 1], k=num_bits)
    alice_bases = choices(['Z', 'X'], k=num_bits)

    # Alice creates and sends qubits
    alice_circuit = cirq.Circuit()

    for bit in range(num_bits):
        encode_gate = encode_gates[alice_key[bit]]
        basis_gate = basis_gates[alice_bases[bit]]
        qubit = qubits[bit]

        alice_circuit.append(encode_gate(qubit))
        alice_circuit.append(basis_gate(qubit))

    # EVE'S ATTACK (intercept-resend if present)
    eve_key = None
    eve_bases = None

    if with_eve:
        # Intercept-resend attack - Eve guesses bases
        eve_bases = choices(['Z', 'X'], k=num_bits)

        # Eve applies her basis gates and measures
        eve_circuit = cirq.Circuit()
        for bit in range(num_bits):
            basis_gate = basis_gates[eve_bases[bit]]
            eve_circuit.append(basis_gate(qubits[bit]))
        eve_circuit.append(cirq.measure(qubits, key='eve_key'))

        sim = cirq.Simulator()
        results = sim.run(alice_circuit + eve_circuit)
        eve_key = results.measurements['eve_key'][0]

        # Eve resends new qubits based on her measurement results
        alice_circuit = cirq.Circuit()
        for bit in range(num_bits):
            encode_gate = encode_gates[eve_key[bit]]
            basis_gate = basis_gates[eve_bases[bit]]
            alice_circuit.append(encode_gate(qubits[bit]))
            alice_circuit.append(basis_gate(qubits[bit]))

    # Bob chooses random bases
    bob_bases = choices(['Z', 'X'], k=num_bits)

    # Bob applies his bases and measures
    bob_circuit = cirq.Circuit()

    for bit in range(num_bits):
        basis_gate = basis_gates[bob_bases[bit]]
        bob_circuit.append(basis_gate(qubits[bit]))

    bob_circuit.append(cirq.measure(qubits, key='bob_key'))

    # Run the full circuit
    full_circuit = alice_circuit + bob_circuit
    sim = cirq.Simulator()
    results = sim.run(full_circuit)
    bob_key = results.measurements['bob_key'][0]

    #Compare bases (keep only matching bases)
    final_alice_key = []
    final_bob_key = []
    final_eve_key = [] if with_eve else None

    for bit in range(num_bits):
        if alice_bases[bit] == bob_bases[bit]:
            final_alice_key.append(alice_key[bit])
            final_bob_key.append(bob_key[bit])
            if with_eve and eve_key is not None:
                final_eve_key.append(eve_key[bit])

    return {
        'alice_original': alice_key,
        'bob_original': bob_key,
        'eve_original': eve_key,
        'alice_final': final_alice_key,
        'bob_final': final_bob_key,
        'eve_final': final_eve_key,
        'alice_bases': alice_bases,
        'bob_bases': bob_bases,
        'eve_bases': eve_bases
    }

def run_statistical_test(num_trials: int, num_bits: int) -> dict:
    """Run multiple trials to analyze Eve's detection statistics for Intercept-Resend attack"""

    error_rates = []
    eve_success_rates = []

    for trial in range(num_trials):
        results = run_bb84(num_bits, with_eve=True)

        # Calculate error rate between Alice and Bob
        match_rate = compare_keys(results['alice_final'], results['bob_final'])
        error_rate = 100 - match_rate
        error_rates.append(error_rate)

        # Calculate how much Eve learned
        if results['eve_final']:
            eve_match = compare_keys(results['eve_final'], results['alice_final'])
            eve_success_rates.append(eve_match)

    return {
        'avg_error_rate': np.mean(error_rates),
        'std_error_rate': np.std(error_rates),
        'avg_eve_success': np.mean(eve_success_rates) if eve_success_rates else 0,
        'min_error': min(error_rates),
        'max_error': max(error_rates),
        'all_error_rates': error_rates,
        'all_eve_success_rates': eve_success_rates
    }

# Run statistical tests
num_trials = 100
num_bits = 1000

print(f"\nRunning {num_trials} trials with {num_bits} qubits each (Intercept-Resend Attack)...\n")

print("\n" + "-"*50)
print("INTERCEPT-RESEND ATTACK STATISTICS:")
intercept_stats = run_statistical_test(num_trials, num_bits)
print(f"  Average Alice-Bob Error Rate: {intercept_stats['avg_error_rate']:.1f}% \u00b1 {intercept_stats['std_error_rate']:.1f}%")
print(f"  Eve's Key Success Rate: {intercept_stats['avg_eve_success']:.1f}%")
print("-"*50)


## Expanded Bloch Sphere Visualizations

In [ ]:
import cirq
import cirq_web
import numpy as np
from IPython.display import display, Markdown

def display_bloch_sphere_for_state(state_vector, title):
    display(Markdown(f"#### {title}"))
    display(cirq_web.bloch_sphere.BlochSphere(state_vector=state_vector))

q = cirq.NamedQubit('q')
sim = cirq.Simulator()

# --- State |0> ---
circuit_0 = cirq.Circuit()
state_0 = sim.simulate(circuit_0, qubit_order=[q]).final_state_vector
display_bloch_sphere_for_state(state_0, "State |0>")

# --- State |1> ---
circuit_1 = cirq.Circuit(cirq.X(q))
state_1 = sim.simulate(circuit_1, qubit_order=[q]).final_state_vector
display_bloch_sphere_for_state(state_1, "State |1>")

# --- State |+> (0 in X basis) ---
circuit_plus = cirq.Circuit(cirq.H(q))
state_plus = sim.simulate(circuit_plus, qubit_order=[q]).final_state_vector
display_bloch_sphere_for_state(state_plus, "State |+> (X basis for 0)")

# --- State |-> (1 in X basis) ---
circuit_minus = cirq.Circuit(cirq.X(q), cirq.H(q))
state_minus = sim.simulate(circuit_minus, qubit_order=[q]).final_state_vector
display_bloch_sphere_for_state(state_minus, "State |-> (X basis for 1)")

# --- State prepared with a 45-degree rotation (e.g., Ry(pi/4)) ---
circuit_ry_45 = cirq.Circuit(cirq.ry(np.pi/4)(q))
state_ry_45 = sim.simulate(circuit_ry_45, qubit_order=[q]).final_state_vector
display_bloch_sphere_for_state(state_ry_45, "State after Ry(pi/4) (45 degrees)")

# --- State prepared with a 90-degree rotation (e.g., Ry(pi/2)) ---
circuit_ry_90 = cirq.Circuit(cirq.ry(np.pi/2)(q))
state_ry_90 = sim.simulate(circuit_ry_90, qubit_order=[q]).final_state_vector
display_bloch_sphere_for_state(state_ry_90, "State after Ry(pi/2) (90 degrees)")
